# 139 — Planificación de movimiento y navegación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=139)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — A* a mano

- (a) `f(1,0) = g 1 + h 7 = 8`; `f(0,1) = 1 + 7 = 8`. Ambos en la franja
  óptima.
- (b) Sí existe: `(0,0)→(1,0)→(2,0)→(3,0)→(4,0)→(4,1)→(4,2)→(4,3)→(4,4)`,
  8 movimientos por el borde derecho, cruzando el muro inferior por el hueco
  `x=4` y el superior también por `x=4`.
- (c) Al menos dos familias: por la izquierda (hueco `x=0` y salida por `x=3`
  o `x=4`) y por la derecha (borde `x=4`). Contando variantes del cruce del
  pasillo central hay 3+ caminos óptimos distintos — A* devuelve uno
  cualquiera de ellos según el orden de desempate.


## Solución 2 — Admisibilidad con 8 vecinos

- (a) Manhattan: **no admisible**. Una diagonal cubre dx=dy=1 con coste 1.414,
  pero Manhattan la valora en 2 > 1.414: sobreestima.
- (b) Euclídea: **admisible** — es la distancia mínima geométrica posible.
- (c) Chebyshev: **admisible** — con diagonales de coste ≥1, se necesitan al
  menos `max(|dx|,|dy|)` pasos.
- (d) 2×euclídea: **no admisible** por construcción.

Síntoma de las no admisibles: A* sigue encontrando caminos (y más rápido),
pero puede devolver uno subóptimo sin ninguna señal de error — el peor tipo de
fallo, silencioso.


## Solución 3 — A* vs Dijkstra

Ambos devuelven coste 8. En esta rejilla, Dijkstra expande casi todos los
nodos libres (~19-20) porque sin guía explora por anillos de g; A* con
Manhattan expande ~12-15, concentrados en la banda diagonal hacia la meta. La
ganancia crece con el tamaño del mapa. En el JSON del lab, la semilla fija la
instancia y `evidence` recoge los hechos verificables del episodio de
búsqueda, igual que tu contador de expansiones.


In [ ]:
import heapq
from ai_evolution.labs import run_lab

OBST = {(1,1),(2,1),(3,1),(1,3),(2,3)}
START, GOAL, N = (0,0), (4,4), 5

def search(use_h):
    def h(p):
        return (abs(GOAL[0]-p[0]) + abs(GOAL[1]-p[1])) if use_h else 0
    openq = [(h(START), 0, START)]
    g = {START: 0}
    expanded = 0
    while openq:
        f, gc, p = heapq.heappop(openq)
        if gc > g.get(p, 1e9):
            continue
        expanded += 1
        if p == GOAL:
            return gc, expanded
        x, y = p
        for nx, ny in ((x+1,y),(x-1,y),(x,y+1),(x,y-1)):
            q = (nx, ny)
            if 0 <= nx < N and 0 <= ny < N and q not in OBST and gc+1 < g.get(q, 1e9):
                g[q] = gc + 1
                heapq.heappush(openq, (gc+1+h(q), gc+1, q))

print("A*      :", search(True))
print("Dijkstra:", search(False))
result = run_lab("search", seed=139)
print(result["kind"], "->", len(result["evidence"]), "evidencias")


## Solución 4 — Ventana dinámica

- (a) `Δv = a·Δt = 0.5·0.2 = 0.1` ⇒ rango alcanzable `[0.4, 0.6] m/s`: esa es
  la ventana dinámica.
- (b) Frenar en 0.3 m exige `v² ≤ 2·a·d = 2·0.5·0.3 = 0.3` ⇒
  `v_max = √0.3 ≈ 0.548 m/s`.
- (c) No: 0.5 < 0.548, la velocidad actual aún permite frenar. Pero las
  candidatas de la ventana por encima de 0.548 (p. ej. 0.6) sí deben
  descartarse como inadmisibles — exactamente el filtro de colisión de DWA.


In [ ]:
a, dt, v, d = 0.5, 0.2, 0.5, 0.3
ventana = (v - a*dt, v + a*dt)
v_max = (2*a*d) ** 0.5
print(f"ventana={ventana}, v_max_frenado={v_max:.3f}")
print("descartar 0.6:", 0.6 > v_max, "| mantener 0.5:", v <= v_max)
